# 10 · Functional Programming Tools

**TL;DR** — treat functions as data and chain them into pipelines.

```
  [1,2,3,4,5,6]                              data
       │
       ▼  map(x → x²)                        transform every item
  [1,4,9,16,25,36]
       │
       ▼  filter(x > 10)                     keep some items
  [16,25,36]
       │
       ▼  reduce(+)                          collapse to one value
       77
```

**Agenda**: lambda → higher-order functions → map → filter → reduce →
sorted/any/all → walrus → generators → decorators

## 1. Lambda — anonymous one-expression functions

`lambda params: expression` — the expression's value is auto-returned.

| | `def` | `lambda` |
|---|---|---|
| Name | yes | no (anonymous) |
| Body | many statements | ONE expression |
| Use | reusable logic | throwaway, passed to HOFs |

In [1]:
square = lambda x: x ** 2        # for demo only — in real code use def
print(square(5))

add = lambda a, b: a + b
print(add(2, 3))

25
5


In [2]:
# lambda + ternary
bigger = lambda a, b: a if a > b else b
print(bigger(4, 9))

is_even = lambda x: 'even' if x % 2 == 0 else 'odd'
print(is_even(7))

9
odd


In [3]:
# check: 'a' in a string
starts_a = lambda s: s[0] == 'a'
print(starts_a('apple'), starts_a('banana'))

True False


## 2. Higher-Order Functions (HOF)

A function that **takes** a function and/or **returns** one. You already met
`make_power` (returns) — here's the taking kind:

In [4]:
def transform(f, L):
    """Apply f to every item of L."""
    output = []
    for item in L:
        output.append(f(item))
    return output

L = [1, 2, 3, 4, 5]
print(transform(lambda x: x ** 2, L))
print(transform(lambda x: x ** 3, L))

[1, 4, 9, 16, 25]
[1, 8, 27, 64, 125]


## 3. `map(f, iterable)` — transform every item

Returns a lazy map object → wrap in `list()` to see it.

In [5]:
L = [1, 2, 3, 4, 5]
print(list(map(lambda x: x * 2, L)))
print(list(map(lambda x: x ** 2, L)))

[2, 4, 6, 8, 10]
[1, 4, 9, 16, 25]


In [6]:
# label odd/even
print(list(map(lambda x: 'even' if x % 2 == 0 else 'odd', L)))

['odd', 'even', 'odd', 'even', 'odd']


In [7]:
# extract a field from a list of dicts
users = [
    {'name': 'ravi', 'age': 45},
    {'name': 'asha', 'age': 33},
]
print(list(map(lambda u: u['name'], users)))

['ravi', 'asha']


In [8]:
# two iterables in lockstep
print(list(map(lambda a, b: a + b, [1, 2, 3], [10, 20, 30])))

[11, 22, 33]


## 4. `filter(predicate, iterable)` — keep items where predicate is True

In [9]:
L = [3, 4, 5, 6, 7]
print(list(filter(lambda x: x > 4, L)))

[5, 6, 7]


In [10]:
fruits = ['apple', 'guava', 'cherry', 'avocado']
print(list(filter(lambda f: f.startswith('a'), fruits)))

['apple', 'avocado']


## 5. `functools.reduce(f, iterable)` — collapse to one value

```
 reduce(+, [1,2,3,4,5]):  1+2 → 3+3 → 6+4 → 10+5 → 15
```

In [11]:
import functools

print(functools.reduce(lambda a, b: a + b, [1, 2, 3, 4, 5]))

15


In [12]:
# find the max with reduce
print(functools.reduce(lambda a, b: a if a > b else b, [23, 11, 45, 10, 1]))

45


In [13]:
# the full pipeline from the TL;DR diagram
data = [1, 2, 3, 4, 5, 6]
squared = map(lambda x: x ** 2, data)
big = filter(lambda x: x > 10, squared)
total = functools.reduce(lambda a, b: a + b, big)
print(total)

77


## 6. Friends: `sorted(key=)`, `any`, `all`

In [14]:
users = [
    {'name': 'ravi', 'age': 45},
    {'name': 'asha', 'age': 33},
    {'name': 'meera', 'age': 51},
]
print(sorted(users, key=lambda u: u['age']))
print(max(users, key=lambda u: u['age'])['name'])

[{'name': 'asha', 'age': 33}, {'name': 'ravi', 'age': 45}, {'name': 'meera', 'age': 51}]
meera


In [15]:
nums = [2, 4, 6, 7]
print(any(x % 2 for x in nums))   # at least one odd?
print(all(x > 0 for x in nums))   # all positive?

True
True


## 7. Walrus Operator `:=` (3.8+)

Assign **inside** an expression — compute once, use twice.

In [16]:
values = [3, 91, 12]
if (biggest := max(values)) > 50:
    print('outlier:', biggest)

outlier: 91


In [17]:
# classic use: filter on an expensive computed value, then reuse it
words = ['hi', 'hello', 'hey', 'greetings']
print([(w, n) for w in words if (n := len(w)) > 3])

[('hello', 5), ('greetings', 9)]


## 8. Generators — lazy sequences

A function with `yield` **pauses and resumes** instead of returning once.
Values are produced one at a time — nothing is stored.

```
  list:      [0, 1, 4, 9, ... 999²]   all in memory at once
  generator:  0 → 1 → 4 → 9 → ...     one value alive at a time
```

In [18]:
def squares_up_to(n):
    for i in range(n):
        yield i ** 2          # pause here, hand out one value

gen = squares_up_to(5)
print(gen)                    # a generator object, nothing computed yet
print(next(gen), next(gen), next(gen))   # pull values on demand
print(list(gen))              # drain the rest

<generator object squares_up_to at 0x10b1c7b90>
0 1 4
[9, 16]


In [19]:
# generator EXPRESSION — comprehension with () instead of []
gen = (i ** 2 for i in range(1, 6))
print(sum(gen))

55


In [20]:
# memory proof
import sys
as_list = [i for i in range(1_000_000)]
as_gen = (i for i in range(1_000_000))
print('list:', sys.getsizeof(as_list), 'bytes')
print('gen :', sys.getsizeof(as_gen), 'bytes')

list: 8448728 bytes
gen : 200 bytes


In [21]:
# ❌ generators are one-shot — drained means done → StopIteration
gen = (i for i in range(3))
list(gen)          # drains it
next(gen)

StopIteration: 

## 9. Decorators — wrap a function to add behavior

A decorator is a HOF that takes a function and returns an **upgraded** version.
`@name` is sugar for `func = name(func)`.

```
           ┌── decorator ─────────────┐
  call ──► │ before... ┌──────────┐   │
           │           │ original │   │ ──► result
           │           └──────────┘   │
           │              ...after    │
           └──────────────────────────┘
```

In [22]:
import functools

def announce(func):
    @functools.wraps(func)              # keep func's name & docstring
    def wrapper(*args, **kwargs):
        print(f'→ calling {func.__name__}{args}')
        result = func(*args, **kwargs)
        print(f'← {func.__name__} returned {result}')
        return result
    return wrapper

@announce
def add(a, b):
    """Add two numbers."""
    return a + b

print(add(2, 3))

→ calling add(2, 3)
← add returned 5
5


In [23]:
# thanks to functools.wraps, identity survives
print(add.__name__, '—', add.__doc__)

add — Add two numbers.


In [24]:
# a useful one: time any function
import time

def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f'{func.__name__} took {time.perf_counter() - start:.4f}s')
        return result
    return wrapper

@timed
def slow_sum(n):
    return sum(range(n))

print(slow_sum(10_000_000))

slow_sum took 0.0754s
49999995000000


In [25]:
# decorator WITH arguments — one more layer
def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(times=3)
def hello():
    print('hello!')

hello()

hello!
hello!
hello!


---
## Recap

| Concept | One-liner |
|---|---|
| lambda | one-expression anonymous function |
| HOF | takes/returns functions |
| `map` / `filter` / `reduce` | transform / keep / collapse |
| `sorted(key=)` | sort anything by a lambda |
| `:=` | assign inside an expression |
| generator | `yield` — lazy, one-shot, tiny memory |
| decorator | `@f` wraps a function with extra behavior |